# 02 Baseline Win/Loss Analysis

Use synthetic data to demonstrate required columns and descriptive win/loss cuts.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

rng = np.random.default_rng(42)
n = 80
df = pd.DataFrame({
    'deal_id': [f'D{i:03d}' for i in range(n)],
    'product_type': rng.choice(['RCF', 'Term Loan', 'Guarantee'], n),
    'rating': rng.choice(['A', 'BBB', 'BB'], n),
    'tenor_months': rng.choice([12, 24, 36, 60], n),
    'margin_above_floor_bps': rng.normal(45, 25, n).round(1),
})
score = 0.3 + 0.006 * df['margin_above_floor_bps'] - 0.05 * (df['rating'] == 'BB')
prob = 1 / (1 + np.exp(-score))
df['deal_outcome'] = np.where(rng.random(n) < prob, 'won', 'lost')
df.head()

In [ ]:
df['tenor_bucket'] = pd.cut(df['tenor_months'], bins=[0, 12, 36, 120], labels=['short', 'medium', 'long'])
df['margin_bucket'] = pd.cut(df['margin_above_floor_bps'], bins=[-999, 0, 30, 70, 999], labels=['below_floor', 'guardrail', 'recommended', 'stretch'])
df['won_flag'] = (df['deal_outcome'] == 'won').astype(int)

cuts = ['product_type', 'rating', 'tenor_bucket', 'margin_bucket']
{cut: df.groupby(cut, observed=True)['won_flag'].agg(['count', 'mean']) for cut in cuts}

For real data, the minimum required fields are product or facility type, rating, tenor, margin above floor, and a commercial won/lost outcome. Competitor pricing columns should not be used as model features.